# Module 2 — Les fondamentaux du machine learning

**CONSEIL :** Sauvegardez une copie de ce notebook dans votre Drive avant de commencer !

**Objectifs d'apprentissage**
- Comprendre ce qu'est une **feature** (et pourquoi il en faut)
- Savoir **nettoyer** un dataset sans pleurer
- Faire un **train/test split** sans tricher
- Diagnostiquer un **overfitting** à l'œil nu
- Saisir le **compromis biais-variance** (le seul vrai mystère du ML)

## Le running example : Adult Census Income

Un classique du ML : ~48 000 individus du recensement américain de 1994, 14 variables socio-démographiques, et une cible binaire : **revenu annuel > 50 k$ ?**

Pour les démographes, sociologues du travail et adeptes des régressions logistiques, c'est un terrain connu. Pour les autres : bienvenue dans le monde merveilleux où l'on prédit si quelqu'un gagne plus de 50 000 dollars en regardant son âge et sa profession.

(On notera au passage que le seuil de 50 k$ en 1994 est devenu… nettement moins glamour avec l'inflation. Mais le dataset reste un excellent terrain de jeu pédagogique.)

## Setup

In [ ]:
# !pip install -q scikit-learn pandas matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

In [ ]:
# Téléchargement (mis en cache après la première fois)
ds = fetch_openml("adult", version=2, as_frame=True)
df = ds.frame.copy()
df.head()

## 2.1 La notion de feature

Une **feature** (variable explicative, prédicteur…) est une colonne dont le modèle se sert pour faire sa prédiction.

Dans Adult :

| Type        | Exemples                                              |
| :-:         | -                                                     |
| Numérique   | `age`, `hours-per-week`, `education-num`              |
| Catégorielle| `workclass`, `education`, `marital-status`, `sex`, `race` |

Les modèles **adorent** le numérique, **tolèrent mal** le catégoriel brut. D'où la nécessité d'**encoder** les variables catégorielles : par exemple, transformer une colonne `sex` avec les valeurs `Male`/`Female` en deux colonnes 0/1 (c'est le « one-hot encoding »).

In [ ]:
# Aperçu des types
df.dtypes

In [ ]:
# Distribution de la cible
df["class"].value_counts(normalize=True)

### Hack Time

Dans la cellule ci-dessous :
- Affichez le nombre de modalités uniques de chaque colonne catégorielle
- Identifiez la colonne qui sera la plus pénible à encoder (la championne du nombre de modalités)
- Réfléchissez : dans **votre** domaine de recherche, quelles features seraient pertinentes pour prédire un comportement ou une appartenance ?

In [ ]:
# Votre code ici

## 2.2 Nettoyage et préparation

Règle d'or : **garbage in, garbage out**. Un modèle entraîné sur des données pourries fait des prédictions pourries (et a souvent l'air confiant en le faisant).

Trois grands chantiers :
- Les **valeurs manquantes** (Adult code ses NA en `?`, parce que pourquoi pas)
- Les **outliers** (vraies erreurs ou vrais individus extrêmes ?)
- La **normalisation** des numériques (certains modèles y sont très sensibles, d'autres s'en fichent)

In [ ]:
# Adult code ses valeurs manquantes par '?'
df = df.replace("?", pd.NA)
df.isna().sum()

### Hack Time

- Combien de colonnes ont des valeurs manquantes ? Combien de lignes sont concernées au total ?
- Choisissez une stratégie : suppression des lignes incomplètes, ou imputation par le mode (la valeur la plus fréquente) pour les catégorielles
- Implémentez-la (un `dropna()` ou un `fillna(df.mode().iloc[0])` font l'affaire pour démarrer)

In [ ]:
# Votre code ici

## 2.3 Train / test split

L'objectif d'un modèle ML n'est **pas** de bien marcher sur les données d'entraînement. C'est de **généraliser** à des données non vues.

D'où la nécessité de **séparer** les données en deux : on entraîne le modèle sur 80 % (le « train ») et on mesure sa qualité sur les 20 % restants (le « test »). Le modèle n'a jamais vu ces 20 % pendant l'entraînement, donc s'il s'en sort bien, on peut espérer qu'il marchera aussi sur de vraies nouvelles données.

> Petit raffinement : on **stratifie** sur la cible (la colonne `class`) pour que le test set ait à peu près la même proportion de `>50K` que le train set.

In [ ]:
from sklearn.model_selection import train_test_split

# TODO en séance : remplacer par les colonnes réellement utilisées
# X = df.drop(columns=["class"])
# y = df["class"]
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, stratify=y, random_state=42
# )
# X_train.shape, X_test.shape

## 2.4 Overfitting

Le **piège classique du ML** : le modèle apprend ses données par cœur, performe à 100 % sur le train, et plante royalement sur le test.

Symptôme : précision sur le train >> précision sur le test.

L'**arbre de décision** est la machine à fabriquer de l'overfitting : laissé sans contrainte sur sa profondeur, il mémorise littéralement les données d'entraînement.

(Un arbre de décision, c'est une suite de questions en cascade : « âge > 30 ? Oui → diplôme ? Non → revenu > 50K ? »… Plus l'arbre est profond, plus il pose de questions, plus il mémorise.)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Squelette : entraîner des arbres de profondeur croissante et observer la divergence
# depths = [1, 3, 5, 10, 20, None]
# train_scores, test_scores = [], []
# for d in depths:
#     clf = DecisionTreeClassifier(max_depth=d, random_state=42)
#     clf.fit(X_train, y_train)
#     train_scores.append(clf.score(X_train, y_train))
#     test_scores.append(clf.score(X_test, y_test))
#
# plt.plot(depths, train_scores, "o-", label="train")
# plt.plot(depths, test_scores, "o-", label="test")
# plt.xlabel("Profondeur de l'arbre"); plt.ylabel("Précision"); plt.legend()

### Hack Time

- À partir de quelle profondeur l'overfitting commence-t-il à se voir ?
- Si on retire `max_depth=None`, que se passe-t-il ?
- Quelle profondeur maximise la **performance en test** ? (C'est ce qu'on appelle le *sweet spot*.)

## 2.5 Biais-variance : le seul vrai mystère

Tout modèle est tiraillé entre deux écueils :

- **Biais élevé** : modèle trop simple, il rate les vraies tendances (sous-apprentissage)
- **Variance élevée** : modèle trop sensible aux données particulières du train (sur-apprentissage)

Visuellement, c'est la fameuse courbe en U : trop simple → erreur élevée ; trop complexe → erreur élevée. Le bon modèle est au creux.

| Modèle | Biais | Variance |
|---|---|---|
| Régression logistique | Élevé | Faible |
| Arbre profond | Faible | Élevé |
| Random Forest (forêt d'arbres) | ↘ | ↘ |
| LLM pré-entraîné | Très faible | Énorme — compensée par un pré-entraînement massif |

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Random Forest = une forêt de plein d'arbres de décision dont on moyenne les prédictions.
# C'est la recette empirique qui marche le mieux sur ce type de données tabulaires.

# Squelette pour comparer 3 modèles sur Adult
# models = {
#     "LogReg (biais ↑, variance ↓)": LogisticRegression(max_iter=1000),
#     "Arbre profond (biais ↓, variance ↑)": DecisionTreeClassifier(max_depth=None, random_state=42),
#     "Random Forest (compromis)": RandomForestClassifier(n_estimators=200, random_state=42),
# }
# for name, m in models.items():
#     m.fit(X_train, y_train)
#     print(f"{name:50s}  train={m.score(X_train, y_train):.3f}  test={m.score(X_test, y_test):.3f}")

### Hack Time

- Quel modèle a le plus gros écart train/test ? C'est votre vainqueur de la variance.
- Quel modèle a la pire performance globale ? C'est probablement votre vainqueur du biais.
- Random Forest gagne presque toujours en pratique. Pourquoi ? (Indice : il fait *plein* d'arbres profonds, et moyenne. C'est de la variance qu'on tue par agrégation.)

## Récap module 2

Ces 5 concepts (features · nettoyage · split · overfitting · biais-variance) structurent **tout** le machine learning.
Y compris ce qui se passe dans les LLMs : ils ont une variance vertigineuse, compensée par un pré-entraînement vertigineux.

→ Direction le **module 3** : on retourne au texte pour comprendre comment il devient des **tokens**, le carburant des LLMs.